# 03a — Y2Y corridor-wide optimization

Runs prioritizr on the **full** aligned stack (no crop) — the original corridor-wide analysis,
now over the shared `prioritizr_core.R` engine. All parameters come from
`config.ANALYSES["y2y"]` (which mirrors the module defaults), carried via `manifest.json`
(cell 1 refreshes it). Outputs → `output_data/iter6_y2y/`.

**Configure a run:** edit `config.ANALYSES["y2y"]`, then run this notebook.
**Kernel:** `R (y2y)`. Run cell-by-cell; Ethan runs, Claude never executes.

In [1]:
# ---- Setup: shared engine + this analysis' key + manifest refresh --------
source("prioritizr_core.R")            # pr_* functions (crop/mask, lock-in, weights, solve)
ANALYSIS <- "y2y"            # <-- the ONLY line that differs between 03a / 03b / 03c
PROJ <- normalizePath(getwd())         # run from the project root

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)   # regenerate manifest.json from config for THIS
ctx   <- pr_setup(mpath, PROJ)                 # analysis (stops on failure); print the banner

manifest refreshed from config.py (analysis=y2y)
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=1e-05
outputs -> output_data/iter6_y2y


In [2]:
# ---- Ingest the stack + crop to the ROI + normalize (window set in config.ANALYSES) ----
ctx <- modifyList(ctx, pr_ingest(ctx))

ingested 48 features (8 continuous + 40 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 48 features to total=100000 each (scale-invariant conditioning)


In [3]:
# ---- Planning units + lock-in + feasibility check ----
ctx <- modifyList(ctx, pr_planning_units(ctx))

planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget


In [4]:
# ---- Feature weights (+ any per-analysis up-weighting) ----
ctx <- modifyList(ctx, pr_weights(ctx))

weights: 8 continuous @ 1.0 ; 40 EFG @ 0.0250 (EFG group total = 1.0)


In [5]:
# ---- Spatial-penalty matrices (built only for penalties > 0) ----
ctx <- modifyList(ctx, pr_penalty_matrices(ctx))

neighbor penalty ON (1e-05): binary rook adjacency derived from the PU raster
penalties -> connectivity=0 | boundary=0 | neighbor=1e-05  (0 = off)


In [6]:
# ---- Build the conservation problem ----
bp <- pr_build_problem(ctx); ctx$p <- bp$p; ctx$solve_params <- bp$solve_params

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties: 
││└•1:          neighbor penalties (`penalty` = 0.00001, …)
│├•features:
││├•targets:    relative targets (all equal to 1)
││└•weights:    continuous values (between 0.025 and 1)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      highs solver (`gap` = 0.1, `time_limit` = 43200, …)
# ℹ Use `summary(...)` to see complete formulation.



In [7]:
# ---- Solve (Ethan runs; heavy) -- HiGHS single solution / Gurobi portfolio ----
# A run that hits the time limit returns an INFEASIBLE point (area > budget) -- discard it.
sv <- pr_solve(ctx); ctx$s <- sv$s; ctx$timing <- sv$timing; ctx$n_sol <- sv$n_sol

LP has 4978469 rows; 3762172 cols; 26933196 nonzeros

Coefficient ranges:

  Matrix  [1e-06, 1e+05]

  Cost    [1e-05, 1e+00]

  Bound   [1e+00, 1e+00]

  RHS     [1e+05, 4e+05]



Presolving model

4229265 rows, 3211751 cols, 22764134 nonzeros 5s

4198797 rows, 3181079 cols, 9479273 nonzeros 28s

Presolve reductions: rows 4198797(-779672); columns 3181079(-581093); nonzeros 9479273(-17453923) 

Solving the presolved LP

IPX model has 4198797 rows, 3181079 columns and 9479273 nonzeros

Input
    Number of variables:                                3181079
    Number of free variables:                           0
    Number of constraints:                              4198797
    Number of equality constraints:                     0
    Number of matrix entries:                           9479273

    Matrix range:                                       [1e+00, 1e+00]

    RHS range:                                          [2e+05, 2e+05]

    Objective range:                              

In [8]:
# ---- Per-alternative summaries + selection-frequency map ----
ctx <- modifyList(ctx, pr_summaries(ctx))

  alternative n_selected pct_region n_added_beyond_pa
1      alt_01   381874.2         30            190705


In [9]:
# ---- Write outputs for 04 (portfolio / frequency / representation / run_summary) ----
pr_write_outputs(ctx)

wrote:
  output_data/iter6_y2y/portfolio.tif
  output_data/iter6_y2y/selection_frequency.tif
  output_data/iter6_y2y/portfolio_representation.csv
  output_data/iter6_y2y/run_summary.json
